# 策略执行引擎教程

本教程介绍 open-xquant 的核心管道：**Universe → Signal → PortfolioOptimizer → Rule** 模型。

我们将用一个 SMA 均线交叉策略（SMA10 金叉/死叉 SMA50）作为贯穿全文的示例，逐层拆解每个阶段的工作方式：

- **Indicator** — 路径无关的纯函数计算（向量化），由 Signal 的 `required_indicators` 声明，Engine 自动收集并计算
- **Signal** — 跨 symbol 截面操作，生成方向性预测（向量化）
- **PortfolioOptimizer** — 根据信号和指标数据，计算目标权重
- **Rule** — 路径相关的状态机，读取持仓返回 RuleResult（逐 bar），传入 `engine.run(rules=[...])`
- **Engine** — 串联各阶段，通过三个 Protocol 接口驱动执行

### 三接口架构：一个引擎，三种模式

Engine 不知道自己在运行回测、模拟盘还是实盘——它只依赖三个 Protocol 接口：

| Protocol | 回测 | 模拟盘（未来） | 实盘（未来） |
|----------|------|---------------|-------------|
| `MarketDataProvider` | `LocalMarketDataProvider` | `RealtimeDataProvider` | `RealtimeDataProvider` |
| `Broker` | `SimBroker` | `SimBroker` | `BrokerAdapter` |

同一套策略代码，不改一行，只需更换 Provider 即可在回测、模拟盘、实盘之间切换。

## 1. 安装依赖

执行引擎是 open-xquant 核心功能，无需额外依赖。如需下载真实行情数据：

```bash
pip install open-xquant[yfinance]
```

---
## 2. Indicator — 技术指标

Indicator 是路径无关的纯函数：输入一个 symbol 的 DataFrame，输出等长 Series，引擎负责追加为宽表的新列。

在新架构中，Indicator 不再直接注册到 Strategy 上，而是通过 Signal 的 `required_indicators` 属性声明依赖。Engine 自动收集所有 Signal 依赖的 Indicator 并计算。

SMA（简单移动平均线）是最基础的趋势指标：

In [1]:
import pandas as pd
from oxq.indicators import SMA

# 构造一段模拟行情
dates = pd.bdate_range("2024-01-01", periods=10)
mktdata = pd.DataFrame({
    "close": [100, 102, 101, 105, 108, 107, 110, 112, 109, 115],
}, index=dates)

sma = SMA()
result = sma.compute(mktdata, period=3)

mktdata["sma_3"] = result
print("SMA(3) 计算结果：")
print(mktdata[["close", "sma_3"]])

SMA(3) 计算结果：
            close       sma_3
2024-01-01    100         NaN
2024-01-02    102         NaN
2024-01-03    101  101.000000
2024-01-04    105  102.666667
2024-01-05    108  104.666667
2024-01-08    107  106.666667
2024-01-09    110  108.333333
2024-01-10    112  109.666667
2024-01-11    109  110.333333
2024-01-12    115  112.000000


前 `period - 1` 行是 NaN（滚动窗口不足），这是正常行为。Signal 和 Rule 层会处理这些 NaN。

**关键特性**：
- `compute` 是纯函数 — 不修改输入 DataFrame，不依赖外部状态
- 同一个 SMA 类可以用不同参数注册为多个实例（如 `sma_10` 和 `sma_50`）
- 默认对 `close` 列计算，也可以通过 `column` 参数指定其他列

验证 SMA 满足 Indicator Protocol：

In [2]:
from oxq.core import Indicator

print(f"SMA 满足 Indicator Protocol: {isinstance(SMA(), Indicator)}")
print(f"Indicator name: {SMA().name}")

SMA 满足 Indicator Protocol: True
Indicator name: SMA


---
## 3. Signal — 信号生成

Signal 描述「交易的欲望」——方向性预测，而非交易指令，表现为 True/False 序列。与 Indicator 的关键区别：

| 维度 | Indicator | Signal |
|------|-----------|--------|
| 输入 | 单个 symbol 的 DataFrame | 单个 symbol 的 DataFrame |
| 输出 | 一个 Series（数值） | 一个 Series（布尔/分类） |
| 语义 | 描述市场状态 | 判断交易意图 |

两者都是**逐 symbol 计算**，签名相同：`compute(mktdata: DataFrame, **params) -> Series`。区别在于语义——Indicator 输出连续数值，Signal 输出离散的交易意图。

Signal 通过 `required_indicators` 属性声明自己依赖的 Indicator，Engine 会自动收集并在计算 Signal 之前完成 Indicator 计算。

Crossover 信号检测快线上穿慢线的时刻：

In [ ]:
from oxq.signals import Crossover

# 构造一组含有金叉的数据
dates = pd.bdate_range("2024-01-01", periods=6)
df = pd.DataFrame({
    "close": [100, 98, 97, 99, 102, 105],
    "sma_10": [99, 98, 97, 99, 101, 103],    # 快线
    "sma_50": [100, 100, 100, 100, 100, 100],  # 慢线
}, index=dates)

# Signal 逐 symbol 计算：接收单个 DataFrame，返回单个 Series
crossover = Crossover()
signal_series = crossover.compute(df, fast="sma_10", slow="sma_50")

df["sma_10_x_sma_50"] = signal_series
print("Crossover 信号：")
print(df[["sma_10", "sma_50", "sma_10_x_sma_50"]])

Day 5（2024-01-05）触发了上穿信号：前一天 sma_10(99) <= sma_50(100)，当天 sma_10(101) > sma_50(100)。

**Signal 和 Indicator 的关系**：两者签名相同（逐 symbol 计算），区别在于语义。Indicator 输出连续数值（如移动平均线），Signal 输出离散的交易意图（如 True/False）。跨 symbol 的操作（如排名、权重分配）由 PortfolioOptimizer 负责。

---
## 4. Rule — 交易规则

Rule 是路径相关的——它知道当前持仓和资金状态，逐 bar 执行，返回 `RuleResult`。

在新架构中，Rule 不再属于 Strategy，而是传入 `engine.run(rules=[...])`。Rule 返回的是 `RuleResult`（包含 target_positions、weights 等），而不是直接的 Order。

与 Indicator/Signal 的关键区别：

| 维度 | Indicator / Signal | Rule |
|------|-------------------|------|
| 输入 | 整个时间序列 | **单行**（当前 bar） |
| 状态 | 无状态（纯函数） | **有状态**（读取 Portfolio） |
| 输出 | Series | **RuleResult** |
| 计算模式 | 向量化 | 逐 bar 循环 |

In [ ]:
from oxq.core import Portfolio, Position, RuleResult
from oxq.rules import ExitRule, StopLossRule

# ExitRule: 快线 < 慢线 + 有持仓 → 返回 RuleResult 表示退出意图
exit_rule = ExitRule(fast="sma_10", slow="sma_50")

# 模拟一个 bar 的数据：快线低于慢线
row = pd.Series({"close": 95.0, "sma_10": 97.0, "sma_50": 100.0})
portfolio = Portfolio(
    cash=50_000.0,
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=102.0)},
)

rule_result = exit_rule.evaluate("AAPL", row, portfolio)
print(f"ExitRule 返回: {rule_result}")
print(f"  target_positions: {rule_result.target_positions}")
print(f"  reason: {rule_result.reason}")

In [ ]:
# StopLossRule: 亏损超过阈值 → 返回 RuleResult 表示止损
stop_loss = StopLossRule(threshold=0.05)  # 5% 止损

row_loss = pd.Series({"close": 96.0})  # 96 / 102 = -5.9%，触发止损
portfolio_with_pos = Portfolio(
    cash=50_000.0,
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=102.0)},
)

sl_result = stop_loss.evaluate("AAPL", row_loss, portfolio_with_pos)
print(f"StopLossRule 返回: {sl_result}")
print(f"  target_positions: {sl_result.target_positions}")
print(f"  reason: {sl_result.reason}")

**Rule 返回 RuleResult**：Rule 不再直接返回 Order，而是返回 `RuleResult`，其中包含 `target_positions`（目标仓位）、`weights`（权重覆盖）等信息。Engine 根据 RuleResult 和 PortfolioOptimizer 的输出统一生成订单。

Rules 通过 `engine.run(rules=[...])` 传入，与 Strategy 定义分离。

---
## 5. Strategy — 声明式策略定义

Strategy 将 Universe、Signal、PortfolioOptimizer 组合为一个完整的声明式管道。Indicator 通过 Signal 的 `required_indicators` 自动收集，Rule 在 `engine.run()` 时传入。

```
Universe            → 确定标的池
  ↓
Signal              → 生成信号列（内部通过 required_indicators 声明依赖的 Indicator）
  ↓
PortfolioOptimizer  → 根据信号计算目标权重
```

In [ ]:
from oxq.core import Strategy
from oxq.universe import StaticUniverse
from oxq.indicators import SMA
from oxq.signals import Crossover
from oxq.rules import ExitRule
from oxq.portfolio.optimizers import EqualWeightOptimizer

# 创建 Crossover 信号，并声明它依赖的 Indicator
crossover = Crossover()
crossover.required_indicators = {
    "sma_10": (SMA(), {"period": 10}),
    "sma_50": (SMA(), {"period": 50}),
}

strategy = Strategy(
    name="sma_crossover",
    universe=StaticUniverse(("AAPL",)),
    signals={
        "sma_10_x_sma_50": (crossover, {"fast": "sma_10", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
)

print(f"策略名称: {strategy.name}")
print(f"假设: {strategy.hypothesis}")
print(f"信号: {list(strategy.signals.keys())}")
print(f"组合优化器: {strategy.portfolio.name}")

Strategy 定义是纯声明式的——它描述「做什么」，不关心「怎么做」。Strategy 包含 Universe、Signal 和 PortfolioOptimizer；Rule 在 `engine.run()` 时注入，实现了策略定义与执行规则的分离。同一个 Strategy 对象可以在回测、模拟盘、实盘中执行，代码零修改。

---
## 6. 宽表数据模型

在运行引擎之前，先理解它如何处理数据。`mktdata` 是按 symbol 索引的 DataFrame 集合，各阶段通过**追加列**逐步加宽每个 symbol 的宽表：

```
原始行情              Indicator 后            Signal 后
+-----------+       +---------------+      +------------------+
| open      |       | open          |      | open             |
| high      |       | high          |      | high             |
| low       | ──▶  | low           | ──▶ | low              |
| close     |       | close         |      | close            |
| volume    |       | volume        |      | volume           |
|           |       | sma_10  (新增)|      | sma_10           |
|           |       | sma_50  (新增)|      | sma_50           |
|           |       |               |      | sma_10_x_sma_50  |
+-----------+       +---------------+      +------------------+
```

Indicator 列由 Engine 根据 Signal 的 `required_indicators` 自动收集并计算，无需手动注册。Signal 无需知道 Indicator 的输出格式，只需按列名引用；Rule 同理。所有中间结果在同一张表上可见可查。

---
## 7. 运行引擎

Engine 将各阶段串联执行。它不知道自己在运行回测——它只是通过 Provider 接口获取数据、提交订单、接收成交。

当我们传入 `LocalMarketDataProvider`（历史数据）+ `SimBroker`（模拟撮合），这就等于回测。Rule 通过 `rules` 参数传入 `engine.run()`。

In [7]:
from oxq.data import YFinanceDownloader

# 下载 AAPL 2023-2024 两年数据
downloader = YFinanceDownloader()
path = downloader.download("AAPL", start="2023-01-01", end="2024-12-31")
print(f"数据已保存到: {path}")

数据已保存到: /Users/daodao/.oxq/data/market/AAPL.parquet


In [ ]:
from oxq.core import Engine
from oxq.data import LocalMarketDataProvider
from oxq.trade import SimBroker

# 选择 Provider：历史数据 + 模拟撮合 = 回测模式
market = LocalMarketDataProvider()
sim_broker = SimBroker()

# Rule 在 engine.run() 时传入，不属于 Strategy
exit_rule = ExitRule(fast="sma_10", slow="sma_50")

engine = Engine()
result = engine.run(
    strategy,
    market=market,
    broker=sim_broker,
    rules=[exit_rule],
    start="2023-01-01",
    end="2024-12-31",
)

print(f"总收益率:   {result.total_return():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"最大回撤:   {result.max_drawdown():.2%}")
print(f"交易次数:   {len(result.trades)}")

引擎内部执行流程：

1. **Phase 0 — Universe**：从 `StaticUniverse` 获取标的列表 `["AAPL"]`
2. **Phase 1 — Indicator**：从所有 Signal 的 `required_indicators` 收集 Indicator，对 AAPL 的 DataFrame 调用 `SMA.compute()`，追加 `sma_10`、`sma_50` 两列
3. **Phase 2 — Signal**：逐 symbol 调用 `Crossover.compute(df)`，追加 `sma_10_x_sma_50` 布尔列
4. **Phase 3 — Portfolio + Rule**：PortfolioOptimizer 计算目标权重，逐 bar 遍历并执行 `rules` 列表中的规则，通过 `broker` 提交订单并接收成交

---
## 8. 查看交易记录

`result.trades` 包含所有成交记录（`Fill` 对象）：

In [9]:
if result.trades:
    print(f"{'日期':<25} {'方向':>4}  {'数量':>4}  {'标的':<6} {'成交价':>8}")
    print("-" * 55)
    for fill in result.trades:
        print(
            f"{fill.filled_at:<25} {fill.order.side:>4}  "
            f"{fill.order.shares:>4}  {fill.order.symbol:<6} "
            f"{fill.filled_price:>8.2f}"
        )
else:
    print("无交易记录")

日期                          方向    数量  标的          成交价
-------------------------------------------------------
2023-10-17 00:00:00        BUY   100  AAPL     175.10
2023-10-23 00:00:00       SELL   100  AAPL     171.00
2023-11-10 00:00:00        BUY   100  AAPL     184.48
2024-01-09 00:00:00       SELL   100  AAPL     183.24
2024-01-30 00:00:00        BUY   100  AAPL     186.11
2024-02-05 00:00:00       SELL   100  AAPL     185.75
2024-05-06 00:00:00        BUY   100  AAPL     180.07
2024-08-12 00:00:00       SELL   100  AAPL     216.11
2024-08-19 00:00:00        BUY   100  AAPL     224.42
2024-09-13 00:00:00       SELL   100  AAPL     221.05
2024-09-23 00:00:00        BUY   100  AAPL     224.99
2024-11-11 00:00:00       SELL   100  AAPL     223.01
2024-11-25 00:00:00        BUY   100  AAPL     231.60


---
## 9. 查看宽表

`result.mktdata` 保留了完整的宽表，可以直接观察 Indicator 和 Signal 的计算结果：

In [10]:
df = result.mktdata["AAPL"]
print(f"宽表列: {list(df.columns)}")
print(f"总行数: {len(df)}")
print()

# 显示信号触发点附近的数据
signal_days = df[df["sma_10_x_sma_50"] == True]
print(f"金叉触发次数: {len(signal_days)}")
if not signal_days.empty:
    print()
    print("金叉触发日的宽表数据：")
    print(signal_days[["close", "sma_10", "sma_50", "sma_10_x_sma_50"]])

宽表列: ['open', 'high', 'low', 'close', 'volume', 'sma_10', 'sma_50', 'sma_10_x_sma_50']
总行数: 501

金叉触发次数: 7

金叉触发日的宽表数据：
                 close      sma_10      sma_50  sma_10_x_sma_50
date                                                           
2023-10-17  175.097900  175.806609  175.718195             True
2023-11-10  184.483490  176.160033  174.434605             True
2024-01-30  186.106598  189.313316  188.962170             True
2024-05-06  180.071213  171.079034  170.817614             True
2024-08-19  224.415848  216.855614  216.609894             True
2024-09-23  224.992050  221.085707  220.804152             True
2024-11-25  231.604813  226.674747  226.636324             True


---
## 10. 分阶段执行（Partial Execution）

引擎支持 `run_through` 参数，在任意阶段终止执行。这对逐组件独立评估非常有用——先验证 Indicator 是否合理，再看 Signal 是否有预测力，最后才加入 PortfolioOptimizer 和 Rule。

In [ ]:
# 只执行到 Indicator 阶段
sim_broker_ind = SimBroker()
result_ind = engine.run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_ind,
    start="2023-01-01",
    end="2024-12-31",
    run_through="indicator",
)

df_ind = result_ind.mktdata["AAPL"]
print(f"Indicator 阶段 — 宽表列: {list(df_ind.columns)}")
print(f"交易次数: {len(result_ind.trades)}  (预期为 0)")
print()

# SMA 值预览
print(df_ind[["close", "sma_10", "sma_50"]].tail())

In [ ]:
# 只执行到 Signal 阶段
sim_broker_sig = SimBroker()
result_sig = engine.run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_sig,
    start="2023-01-01",
    end="2024-12-31",
    run_through="signal",
)

df_sig = result_sig.mktdata["AAPL"]
print(f"Signal 阶段 — 宽表列: {list(df_sig.columns)}")
print(f"交易次数: {len(result_sig.trades)}  (预期为 0)")
print()

# 信号统计
n_signals = df_sig["sma_10_x_sma_50"].sum()
print(f"金叉信号总次数: {n_signals}")

---
## 11. 三接口架构详解

Engine 通过三个 Protocol 接口与外部世界交互，这是策略与执行环境解耦的关键：

| Protocol | 职责 | 方法 |
|----------|------|------|
| `MarketDataProvider` | 提供行情数据 | `get_bars()`, `get_latest()` |
| `OrderRouter` | 接收并路由订单 | `submit_order()` |
| `FillReceiver` | 返回成交结果 | `get_fills()` |

SimBroker 同时实现了 OrderRouter 和 FillReceiver 两个 Protocol，因此可以作为 `broker` 参数传入：

In [13]:
from oxq.core import OrderRouter, FillReceiver
from oxq.trade import SimBroker

broker = SimBroker()
print(f"SimBroker 满足 OrderRouter: {isinstance(broker, OrderRouter)}")
print(f"SimBroker 满足 FillReceiver: {isinstance(broker, FillReceiver)}")

SimBroker 满足 OrderRouter: True
SimBroker 满足 FillReceiver: True


未来切换到实盘时，只需替换 Provider，策略代码不变：

```python
# 回测：历史数据 + 模拟撮合
engine.run(strategy, market=LocalMarketDataProvider(),
           broker=sim_broker, rules=[exit_rule], ...)

# 模拟盘（未来）：实时数据 + 模拟撮合
engine.run(strategy, market=RealtimeDataProvider(),
           broker=sim_broker, rules=[exit_rule], ...)

# 实盘（未来）：实时数据 + 真实券商
engine.run(strategy, market=RealtimeDataProvider(),
           broker=live_broker, rules=[exit_rule], ...)
```

---
## 12. 多标的策略

同一套策略定义，换一个 Universe 就变成多标的策略，引擎代码零修改：

In [ ]:
# 下载更多标的
for symbol in ["MSFT", "GOOGL"]:
    downloader.download(symbol, start="2023-01-01", end="2024-12-31")

# 创建带 required_indicators 的 Crossover 信号
crossover_multi = Crossover()
crossover_multi.required_indicators = {
    "sma_10": (SMA(), {"period": 10}),
    "sma_50": (SMA(), {"period": 50}),
}

# 只改 Universe，其余不变
multi_strategy = Strategy(
    name="sma_crossover_multi",
    universe=StaticUniverse(("AAPL", "MSFT", "GOOGL")),
    signals={
        "sma_10_x_sma_50": (crossover_multi, {"fast": "sma_10", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
)

exit_rule_multi = ExitRule(fast="sma_10", slow="sma_50")

sim_broker_multi = SimBroker()
result_multi = engine.run(
    multi_strategy,
    market=LocalMarketDataProvider(),
    broker=sim_broker_multi,
    rules=[exit_rule_multi],
    start="2023-01-01",
    end="2024-12-31",
)

print(f"总收益率:   {result_multi.total_return():.2%}")
print(f"Sharpe Ratio: {result_multi.sharpe_ratio():.2f}")
print(f"最大回撤:   {result_multi.max_drawdown():.2%}")
print(f"交易次数:   {len(result_multi.trades)}")
print()

# 按标的分组显示交易
from collections import Counter
trade_counts = Counter(f.order.symbol for f in result_multi.trades)
for symbol, count in sorted(trade_counts.items()):
    print(f"  {symbol}: {count} 笔交易")

---
## 13. PortfolioOptimizer — 组合优化器

在新架构中，仓位管理由 `PortfolioOptimizer` 负责，替代了旧的 EntryRule 系列。PortfolioOptimizer 根据信号和指标数据，计算每个标的的目标权重。

open-xquant 提供三种优化器：

| 优化器 | 逻辑 | 适用场景 |
|--------|------|----------|
| `EqualWeightOptimizer` | 所有信号标的等权 | 简单均配 |
| `RiskParityOptimizer` | 按波动率倒数加权 | 风险平价 |
| `KellyOptimizer` | Kelly 公式计算最优比例 | 凯利准则 |

In [ ]:
from oxq.portfolio.optimizers import EqualWeightOptimizer, RiskParityOptimizer, KellyOptimizer

# --- EqualWeightOptimizer ---
eq = EqualWeightOptimizer()
weights = eq.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
    indicators={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
)
print(f"EqualWeight — 2 标的: {weights}")

weights_3 = eq.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame(), "GOOGL": pd.DataFrame()},
    indicators={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame(), "GOOGL": pd.DataFrame()},
)
print(f"EqualWeight — 3 标的: {weights_3}")

print()

# --- 无信号时返回全现金 ---
weights_empty = eq.optimize(signals={}, indicators={})
print(f"EqualWeight — 无信号: {weights_empty}")

### 回测对比：不同优化器 + 不同 Rule 组合

用同一套信号，替换 PortfolioOptimizer 和 Rule 组合，对比效果：

In [ ]:
# 公共信号组件
def make_crossover_signal():
    sig = Crossover()
    sig.required_indicators = {
        "sma_10": (SMA(), {"period": 10}),
        "sma_50": (SMA(), {"period": 50}),
    }
    return sig

common_signals = lambda: {
    "sma_10_x_sma_50": (make_crossover_signal(), {"fast": "sma_10", "slow": "sma_50"}),
}

# 三种配置：不同 optimizer + 不同 rules
configs = {
    "等权+退出": {
        "strategy": Strategy(
            name="equal_exit",
            universe=StaticUniverse(("AAPL",)),
            signals=common_signals(),
            portfolio=EqualWeightOptimizer(),
            hypothesis="SMA10 金叉 SMA50 买入，死叉卖出",
        ),
        "rules": [ExitRule(fast="sma_10", slow="sma_50")],
    },
    "等权+止损5%": {
        "strategy": Strategy(
            name="equal_stoploss",
            universe=StaticUniverse(("AAPL",)),
            signals=common_signals(),
            portfolio=EqualWeightOptimizer(),
            hypothesis="SMA10 金叉 SMA50 买入，5%止损",
        ),
        "rules": [ExitRule(fast="sma_10", slow="sma_50"), StopLossRule(threshold=0.05)],
    },
    "等权+无规则": {
        "strategy": Strategy(
            name="equal_norule",
            universe=StaticUniverse(("AAPL",)),
            signals=common_signals(),
            portfolio=EqualWeightOptimizer(),
            hypothesis="SMA10 金叉 SMA50 买入，无退出规则",
        ),
        "rules": [],
    },
}

# 分别回测
results = {}
for label, cfg in configs.items():
    broker = SimBroker()
    r = Engine().run(
        cfg["strategy"],
        market=LocalMarketDataProvider(),
        broker=broker,
        rules=cfg["rules"],
        start="2023-01-01",
        end="2024-12-31",
        initial_cash=100_000.0,
    )
    results[label] = r

# 对比
header = f"{'':>14}" + "".join(f"{label:>14}" for label in results)
print(header)
print("-" * len(header))
for metric, fn in [
    ("总收益率", lambda r: f"{r.total_return():.2%}"),
    ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
    ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
    ("交易次数", lambda r: f"{len(r.trades)}"),
    ("期末总资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
]:
    vals = "".join(f"{fn(r):>14}" for r in results.values())
    print(f"{metric:>14}{vals}")

---
## 小结

本教程覆盖了策略执行管道的核心概念：

| 组件 | 职责 | 计算模式 |
|------|------|----------|
| `SMA` | 计算移动平均线 | 向量化，per symbol |
| `Crossover` | 检测上穿信号 | 向量化，per symbol |
| `EqualWeightOptimizer` | 等权分配目标权重 | 跨 symbol |
| `RiskParityOptimizer` | 风险平价分配权重 | 跨 symbol |
| `KellyOptimizer` | Kelly 公式分配权重 | 跨 symbol |
| `ExitRule` | 快线跌破慢线时退出 | 逐 bar，有状态 |
| `StopLossRule` | 亏损超阈值时止损 | 逐 bar，有状态 |
| `Strategy` | 声明式策略定义（Universe + Signal + PortfolioOptimizer） | — |
| `Engine` | 执行管道（provider-agnostic），接收 `rules` 参数 | — |
| `SimBroker` | 模拟撮合（Broker） | — |
| `RunResult` | 绩效指标 + 交易记录 + 宽表 | — |

**执行管道**：

```
Phase 0: Universe         → 确定标的池（StaticUniverse / FilterUniverse）
Phase 1: Indicator        → 从 Signal.required_indicators 自动收集，向量化计算，追加为宽表新列
Phase 2: Signal           → 逐 symbol 信号生成，追加为宽表新列
Phase 3: Portfolio + Rule → PortfolioOptimizer 计算目标权重 + Rule 逐 bar 评估 → 生成订单 → Broker 提交并接收成交
```

**核心设计原则**：
- **策略定义与执行规则分离** — Strategy 包含 Universe/Signal/PortfolioOptimizer，Rule 在 `engine.run(rules=[...])` 时注入
- **三接口架构** — MarketDataProvider、OrderRouter、FillReceiver
- **一个 Engine，三种模式** — 回测 / 模拟盘 / 实盘只需更换 Provider
- Indicator 和 Signal 都是逐 symbol 计算，签名相同，语义不同
- 跨 symbol 的操作（排名、权重分配）由 PortfolioOptimizer 负责
- Rule 返回 RuleResult，而非直接返回 Order
- 宽表避免层间数据传递的复杂性
- `run_through` 支持逐组件独立评估